# NB-10: VAE Encoder -- Downsampling from Video to Latent Space

## Learning Objectives
1. Trace `Encoder3d`'s 4-level downsampling path with a full shape table showing spatial, temporal, and channel dimensions at each stage
2. Understand `Resample` module internals: `downsample2d` (spatial only) and `downsample3d` (spatial + temporal via `time_conv`), including why temporal stays constant in demo cells
3. See the reparameterization trick: `mu`/`log_var` split from encoder output, the `eps * std + mu` formula, and why it enables backprop through sampling

## Prerequisites
- **NB-09**: CausalConv3d, ResidualBlock, AttentionBlock -- the building blocks used in every encoder level

## Concept Map

```
Encoder3d Data Flow (dim=96, z_dim=16 production)
==================================================
[B, 3, T, H, W]  -- raw video (RGB)
       |
       v  conv1: CausalConv3d(3 -> dim=96)
[B, 96, T, H, W]
       |
       v  Level 0: 2x ResBlock(96->96) + Resample(downsample2d)
[B, 96, T, H/2, W/2]
       |
       v  Level 1: 2x ResBlock(96->192) + Resample(downsample3d)
[B, 192, T, H/4, W/4]     (temporal unchanged w/o feat_cache)
       |
       v  Level 2: 2x ResBlock(192->384) + Resample(downsample3d)
[B, 384, T, H/8, W/8]
       |
       v  Level 3: 2x ResBlock(384->384)  (NO downsample -- last level)
       |
       v  Middle: ResBlock + AttentionBlock + ResBlock
       |
       v  Head: RMS_norm + SiLU + CausalConv3d(384 -> z_dim*2=32)
       |
       v  chunk(2, dim=1): mu=[B,16,...], log_var=[B,16,...]
       |
       v  reparameterize: z = eps * std + mu
[B, 16, T, H/8, W/8]  -- latent
```

**Note on temporal compression:** In production, `downsample3d` levels also apply a temporal stride via `time_conv` using `feat_cache` streaming. In demo cells (no `feat_cache`), temporal dimension stays constant -- this is expected and explained in Section 2.

In [ ]:
# Source: established in NB-01, adapted for wan_video_vae.py
import sys, importlib.util, pathlib, types

# Stub tqdm to suppress progress bars in notebook
_tqdm_stub = types.ModuleType('tqdm')
_tqdm_stub.tqdm = lambda x, **kw: x
sys.modules['tqdm'] = _tqdm_stub

# Walk up from notebook location to find the project root
_here = pathlib.Path().resolve()
PROJECT_ROOT = None
_candidate = _here
for _ in range(10):
    if (_candidate / 'diffsynth' / 'models' / 'wan_video_vae.py').exists():
        PROJECT_ROOT = _candidate
        break
    _parent = _candidate.parent
    if _parent == _candidate:
        break
    _candidate = _parent
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find diffsynth/models/wan_video_vae.py')
print(f'Project root: {PROJECT_ROOT}')

# Load wan_video_vae.py directly via importlib (bypasses broken diffsynth/__init__.py)
_vae_path = PROJECT_ROOT / 'diffsynth' / 'models' / 'wan_video_vae.py'
_spec = importlib.util.spec_from_file_location('diffsynth.models.wan_video_vae', _vae_path)
vae = importlib.util.module_from_spec(_spec)
sys.modules['diffsynth.models.wan_video_vae'] = vae
_spec.loader.exec_module(vae)

from diffsynth.models.wan_video_vae import (
    CausalConv3d, RMS_norm, ResidualBlock, AttentionBlock,
    Resample, Encoder3d, Decoder3d, VideoVAE_, patchify, unpatchify
)
import torch
from einops import rearrange
print('Setup complete.')

## 1. Encoder3d Structure -- Sub-Module Overview

`Encoder3d` takes raw RGB video `[B, 3, T, H, W]` and compresses it to a latent representation. It uses `dim_mult = [1, 2, 4, 4]` to progressively increase channels while spatially downsampling. A key detail for the VAE pipeline: `VideoVAE_` instantiates `Encoder3d` with `z_dim = z_dim * 2` (line 971 in `wan_video_vae.py`), so the encoder head outputs `z_dim*2` channels for the subsequent `mu`/`log_var` split.

| Sub-module | Type | Purpose |
|-----------|------|----------|
| `conv1` | `CausalConv3d(3, dim, 3, padding=1)` | Initial channel expansion: 3 RGB -> dim channels |
| `downsamples` | `nn.Sequential(...)` | 4-level downsample: ResBlocks + Resample at each level |
| `middle` | `Sequential(ResBlock, AttentionBlock, ResBlock)` | Bottleneck with spatial self-attention |
| `head` | `Sequential(RMS_norm, SiLU, CausalConv3d)` | Project to z_dim for final output |

**Source:** `diffsynth/models/wan_video_vae.py`, lines 517-617

In [ ]:
# Source: diffsynth/models/wan_video_vae.py, lines 517-617
# --- DEMO CONFIG (dim=8, z_dim=8 for mu/log_var split, STD-03 compliant -- verified ~4ms on CPU) ---
# VideoVAE_ instantiates Encoder3d with z_dim*2, so we use z_dim=8 here
# to get 8-channel output that can be chunk(2, dim=1) -> mu [4ch] + log_var [4ch]
enc = Encoder3d(
    dim=8,                                    # vs 96 in production
    z_dim=8,                                  # z_dim*2 = 8 -> after chunk: mu,log_var each [4ch]
    dim_mult=[1, 2, 4, 4],                    # SAME as production
    num_res_blocks=1,                         # vs 2 in production
    temperal_downsample=[False, True, True]   # SAME as production
)
# --- PRODUCTION CONFIG (annotation only -- do not run on CPU) ---
# VideoVAE_ uses: Encoder3d(dim=96, z_dim=32, dim_mult=[1,2,4,4], num_res_blocks=2,
#                            temperal_downsample=[False, True, True])
# (z_dim=32 because VideoVAE_(z_dim=16) passes z_dim*2=32 to Encoder3d)

# Inspect structure
print(f'conv1: {enc.conv1}')                    # CausalConv3d(3, 8, kernel_size=3, padding=1)
print(f'downsamples: {len(enc.downsamples)} layers')
print(f'middle: {len(enc.middle)} layers')
print(f'head: {len(enc.head)} layers')
print()
print('downsamples breakdown:')
for i, m in enumerate(enc.downsamples):
    if hasattr(m, 'in_dim'):
        print(f'  [{i}] {type(m).__name__}({m.in_dim}->{m.out_dim})')
    else:
        print(f'  [{i}] {type(m).__name__}(mode={m.mode})')

## 2. Resample Module -- Spatial and Temporal Downsampling

`Resample` handles the spatial reduction between encoder levels. Two modes for downsampling:

- **`downsample2d`**: `ZeroPad2d(0,1,0,1)` + `Conv2d(dim, dim, 3, stride=(2,2))` -- halves H and W only, channels unchanged
- **`downsample3d`**: same spatial path + `time_conv = CausalConv3d(dim, dim, (3,1,1), stride=(2,1,1), padding=0)` -- adds a temporal stride conv

**Critical:** `time_conv` only runs when `feat_cache is not None` (production streaming). In demo cells (no `feat_cache`), `downsample3d` behaves identically to `downsample2d` -- only spatial halving occurs. We demonstrate `time_conv` directly in the next cell to show its temporal striding behavior.

**Source:** `diffsynth/models/wan_video_vae.py`, lines 82-196

In [ ]:
# Source: diffsynth/models/wan_video_vae.py, lines 82-196
rs_2d = Resample(8, 'downsample2d')
rs_3d = Resample(8, 'downsample3d')

print('downsample2d internals:')
print(f'  resample: {rs_2d.resample}')
# Sequential(ZeroPad2d((0,1,0,1)), Conv2d(8, 8, 3, stride=(2,2)))

print('\ndownsample3d internals:')
print(f'  resample (spatial): {rs_3d.resample}')
# Sequential(ZeroPad2d((0,1,0,1)), Conv2d(8, 8, 3, stride=(2,2)))
print(f'  time_conv: {rs_3d.time_conv}')
# CausalConv3d(8, 8, kernel_size=(3,1,1), stride=(2,1,1))

x = torch.randn(1, 8, 5, 8, 8)  # [B, C=8, T=5, H=8, W=8]
with torch.no_grad():
    out_2d = rs_2d(x)
    out_3d = rs_3d(x)

print(f'\ndownsample2d: {list(x.shape)} -> {list(out_2d.shape)}')
# [1, 8, 5, 8, 8] -> [1, 8, 5, 4, 4]  spatial /2, temporal unchanged
print(f'downsample3d: {list(x.shape)} -> {list(out_3d.shape)}')
# [1, 8, 5, 8, 8] -> [1, 8, 5, 4, 4]  spatial /2, temporal ALSO unchanged (no feat_cache!)

assert out_2d.shape == (1, 8, 5, 4, 4), f'Expected (1,8,5,4,4), got {out_2d.shape}'
assert out_3d.shape == (1, 8, 5, 4, 4), f'Expected (1,8,5,4,4), got {out_3d.shape}'
print('\nBoth modes produce same shape in demo (no feat_cache) -- spatial only')

In [ ]:
# In production, time_conv is called separately with feat_cache streaming.
# We can call it directly to demonstrate the temporal stride:
# time_conv: CausalConv3d(8, 8, (3,1,1), stride=(2,1,1), padding=(0,0,0))
# _padding = (0,0,0,0,0,0) -- no causal prepend for this conv
# Output temporal: floor((T - k_t) / stride_t) + 1 = (5-3)//2 + 1 = 2

x_tc = torch.randn(1, 8, 5, 8, 8)  # T=5 needed: (5-3)//2+1 = 2
out_tc = rs_3d.time_conv(x_tc)     # stride=(2,1,1) on temporal axis
print(f'time_conv directly: {list(x_tc.shape)} -> {list(out_tc.shape)}')
# [1, 8, 5, 8, 8] -> [1, 8, 2, 8, 8]  temporal: 5 -> 2
assert out_tc.shape[2] == 2, f'Expected T=2, got {out_tc.shape[2]}'
print('In production (feat_cache streaming): Resample(downsample3d) applies temporal stride.')
print('In demo cells (no feat_cache): only spatial halving occurs -- this is expected.')

## 3. Level-by-Level Shape Trace

The encoder uses `dim_mult = [1, 2, 4, 4]` to compute `dims = [dim * u for u in [1] + dim_mult]`. With `dim=8`: `dims = [8, 8, 16, 32, 32]`. Each pair `(dims[i], dims[i+1])` defines one level's input and output channels.

**Demo Shape Table (dim=8, z_dim=8, input [1,3,5,16,16]):**

| Stage | Tensor Shape | Operation |
|-------|-------------|----------|
| Input | `[1, 3, 5, 16, 16]` | Raw video (5 frames, 16x16) |
| After conv1 | `[1, 8, 5, 16, 16]` | CausalConv3d(3->8, k=3, p=1) |
| After L0 ResBlock | `[1, 8, 5, 16, 16]` | 1x ResidualBlock(8->8) |
| After L0 Resample | `[1, 8, 5, 8, 8]` | downsample2d (spatial /2) |
| After L1 ResBlock | `[1, 16, 5, 8, 8]` | 1x ResidualBlock(8->16) |
| After L1 Resample | `[1, 16, 5, 4, 4]` | downsample3d (spatial /2, temporal unchanged w/o cache) |
| After L2 ResBlock | `[1, 32, 5, 4, 4]` | 1x ResidualBlock(16->32) |
| After L2 Resample | `[1, 32, 5, 2, 2]` | downsample3d (spatial /2, temporal unchanged w/o cache) |
| After L3 ResBlock | `[1, 32, 5, 2, 2]` | 1x ResidualBlock(32->32), no Resample (last level) |
| After Middle | `[1, 32, 5, 2, 2]` | ResBlock + AttentionBlock + ResBlock |
| After Head | `[1, 8, 5, 2, 2]` | RMS_norm + SiLU + CausalConv3d(32->8) (z_dim=8) |

**Production Shape Table (dim=96, z_dim=32, input [B,3,81,512,512]):** (annotation only -- do not run on CPU)

| Stage | Tensor Shape | Operation |
|-------|-------------|----------|
| Input | `[B, 3, 81, 512, 512]` | Raw RGB video |
| After conv1 | `[B, 96, 81, 512, 512]` | CausalConv3d(3->96) |
| After L0 Resample | `[B, 96, 81, 256, 256]` | downsample2d (spatial /2) |
| After L1 Resample | `[B, 192, ~21, 128, 128]` | downsample3d (spatial /2 + temporal streaming ~4x) |
| After L2 Resample | `[B, 384, 21, 64, 64]` | downsample3d (spatial /2 + temporal) |
| After Head | `[B, 32, 21, 64, 64]` | CausalConv3d(384->32, z_dim=32=16*2) |
| After chunk | `mu=[B,16,21,64,64], log_var=[B,16,21,64,64]` | `.chunk(2, dim=1)` |
| Latent z | `[B, 16, 21, 64, 64]` | reparameterize |

In [ ]:
# Run reduced-scale encoder forward pass
x = torch.randn(1, 3, 5, 16, 16)  # [B, C=3, T=5, H=16, W=16]
with torch.no_grad():
    z = enc(x)

print(f'Encoder3d: {list(x.shape)} -> {list(z.shape)}')
# [1, 3, 5, 16, 16] -> [1, 8, 5, 2, 2]  (z_dim=8 channels)
assert z.shape == (1, 8, 5, 2, 2), f'Expected (1,8,5,2,2), got {z.shape}'

# Verify shape progression matches table:
# Spatial: 16 -> 8 (L0 ds2d) -> 4 (L1 ds3d) -> 2 (L2 ds3d) = 16 / 8 = 2
# Temporal: 5 -> 5 -> 5 -> 5 = unchanged (no feat_cache)
# Channels: 3 -> 8 -> 8 -> 16 -> 32 -> 32 -> z_dim=8
print(f'Spatial compression: {x.shape[3]}x{x.shape[4]} -> {z.shape[3]}x{z.shape[4]} (/{x.shape[3]//z.shape[3]}x)')
print(f'Temporal: {x.shape[2]} -> {z.shape[2]} (unchanged without feat_cache)')
print(f'Channels: {x.shape[1]} RGB -> {z.shape[1]} (z_dim=8 for mu/log_var split)')

## 4. Reparameterization -- Sampling from the Latent Distribution

The encoder outputs `z_dim` channels (which `VideoVAE_` sets to `z_dim*2` so it's actually `2 * z_dim` of the VAE). These are split into `mu` and `log_var` via `.chunk(2, dim=1)`. The reparameterization trick: instead of sampling `z` directly from `N(mu, sigma^2)` (which blocks gradient flow), compute:

```
z = mu + eps * exp(0.5 * log_var)
```

where `eps ~ N(0, 1)`. Gradients flow through `mu` and `log_var` (the encoder's outputs) because `eps` is treated as a constant drawn independently of the network parameters.

In `VideoVAE_`, the mu/log_var split happens at line 1001: `mu, log_var = self.conv1(out).chunk(2, dim=1)` where `self.conv1 = CausalConv3d(z_dim*2, z_dim*2, 1)` is a 1x1x1 pointwise conv applied to the encoder's output before splitting.

**Source:** `diffsynth/models/wan_video_vae.py`, lines 1036-1039 (`reparameterize`), line 1001 (`mu`/`log_var` split)

In [ ]:
# Source: diffsynth/models/wan_video_vae.py, lines 1036-1039
def reparameterize(mu, log_var):
    std = torch.exp(0.5 * log_var)  # std = sqrt(var) = exp(0.5 * log(var))
    eps = torch.randn_like(std)     # sample from N(0,1) -- NOT a function of parameters
    return eps * std + mu           # z ~ N(mu, std^2) -- gradients flow through mu and std

# Demonstrate mu/log_var split from encoder output:
# Source: wan_video_vae.py, line 1001: mu, log_var = self.conv1(out).chunk(2, dim=1)
enc_output = z  # [1, 8, 5, 2, 2] -- z_dim=8 channels from encoder head
mu, log_var = enc_output.chunk(2, dim=1)  # split along channel axis
print(f'Encoder output:  {list(enc_output.shape)}')  # [1, 8, 5, 2, 2]
print(f'mu:              {list(mu.shape)}')           # [1, 4, 5, 2, 2]
print(f'log_var:         {list(log_var.shape)}')      # [1, 4, 5, 2, 2]

latent_z = reparameterize(mu, log_var)
print(f'Latent z:        {list(latent_z.shape)}')     # [1, 4, 5, 2, 2]
assert latent_z.shape == mu.shape, f'Shape mismatch: {latent_z.shape} != {mu.shape}'
assert latent_z.shape == (1, 4, 5, 2, 2), f'Expected (1,4,5,2,2), got {latent_z.shape}'

# In production VideoVAE_.encode: a learned 1x1x1 CausalConv3d is applied before split
# conv1 = CausalConv3d(z_dim*2, z_dim*2, 1) -- pointwise, no spatial mixing
print('\nIn production: mu, log_var = self.conv1(encoder_out).chunk(2, dim=1)')
print('conv1 is CausalConv3d(z_dim*2, z_dim*2, 1) -- pointwise mixing before split')

## Summary

### Key Takeaways
- **`Encoder3d`** compresses `[B, 3, T, H, W]` video to `[B, z_dim*2, T, H/8, W/8]` latent via 4 levels with `dim_mult=[1, 2, 4, 4]`
- **`Resample`**: `downsample2d` = spatial-only (Level 0); `downsample3d` = spatial + temporal (Levels 1, 2), but temporal stride only applies with `feat_cache` streaming in production
- **Middle block**: `ResBlock + AttentionBlock + ResBlock` provides bottleneck spatial self-attention
- **Reparameterization**: `z = mu + eps * std` where `eps ~ N(0,1)` enables gradient flow through the sampling operation; the stochastic part `eps` is not a function of model parameters
- **`VideoVAE_` instantiation detail**: `VideoVAE_` calls `Encoder3d(dim, z_dim*2, ...)`, so the encoder's head outputs `z_dim*2` channels for the `mu`/`log_var` split

### Source References

| Symbol | Location |
|--------|----------|
| `Encoder3d` | `diffsynth/models/wan_video_vae.py`, line 517 |
| `Resample` | `diffsynth/models/wan_video_vae.py`, line 82 |
| `VideoVAE_.reparameterize` | `diffsynth/models/wan_video_vae.py`, line 1036 |
| `VideoVAE_.encode` (mu/log_var split) | `diffsynth/models/wan_video_vae.py`, line 984 |
| `CausalConv3d` | NB-09, `wan_video_vae.py`, line 33 |
| `ResidualBlock` | NB-09, `wan_video_vae.py`, line 267 |
| `AttentionBlock` | NB-09, `wan_video_vae.py`, line 304 |

## Exercises

### Exercise 1 -- Trace with different spatial input
Change the input to `[1, 3, 5, 32, 32]` and predict the latent shape before running. Remember the encoder applies 3 spatial downsamples (8x total). Verify with `enc(torch.randn(1, 3, 5, 32, 32))`. What are the output spatial dimensions?

### Exercise 2 -- Inspect Resample conv weights
For `Resample(8, 'downsample2d')`, print `rs_2d.resample[1].weight.shape` (the Conv2d weight). Verify the stride is `(2, 2)` by printing `rs_2d.resample[1].stride`. Why does `downsample2d` use `ZeroPad2d((0, 1, 0, 1))` before the strided conv instead of a symmetric padding?

### Exercise 3 -- Reparameterization determinism
Call `reparameterize(mu, log_var)` twice with `torch.manual_seed(42)` before each call. Are the results identical? Then call without a seed. Are those two calls identical? Explain why the seed matters for reproducibility in VAE training.